# Elevation zone evaluation
---
Compare binning of low, mid, high elevations in paper against CBRFC modeling units for each basin  

Toss in SNOTEL elevations when you get the chance  

*J. Michelle Hu  
University of Utah  
September 2025* 

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
import geopandas as gpd
import xarray as xr
import rioxarray
# import hvplot.xarray

from pathlib import PurePath

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

import plot_spatial_comparison as psc

from rasterio.enums import Resampling
import seaborn as sns
sns.set_palette('icefire')

In [ ]:
# Turn off warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
basins = ['blue', 'animas', 'yampa']

In [ ]:
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'
# Grab clips of basin DEMs
dem_fns = [h.fn_list(script_dir, f'{basin}_setup/output_100m/temp/clipped_dem.tif')[0] for basin in basins]
print(dem_fns)

# Read them in
dem_list = [h.load(dem_fn) for dem_fn in dem_fns]
# Rename them with the basin
for dem, basin in zip(dem_list, basins):
    dem.name = basin
print(len(dem_list))

# Plot them
fig, axa = plt.subplots(1, 3, figsize=(15, 5))
for dem, ax in zip(dem_list, axa.flatten()):
    h.plot_one(dem, specify_ax=(fig, ax), title=dem.name)

In [ ]:
# Read in CBRFC stuff
cbrfc_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/CBRFC_SNOW17/basin_zones'
cbrfc_fns = [h.fn_list(cbrfc_dir, f'*{basin}*gpkg')[0] for basin in basins]

cbrfc_poly_list = [gpd.read_file(cbrfc_fn) for cbrfc_fn in cbrfc_fns]
cbrfc_poly_list[0]

In [ ]:
# Plot them, coloring by zoneNum
fig, axa = plt.subplots(1, 3, figsize=(15, 5))
for poly, ax in zip(cbrfc_poly_list, axa.flatten()):
    poly.plot(ax=ax, column='zoneNum', legend=True, cmap='viridis_r', vmin=1, vmax=3, ec='k', linewidth=0.15)

In [ ]:
# Warp cbrfc domains to match dem and 
# Crop the DEMs to the bounds of the CBRFC polygons
cbrfc_warped_list = []
dem_cropped_list = []
for cbrfc_poly, dem in zip(cbrfc_poly_list, dem_list):
    cbrfc_warped = cbrfc_poly.to_crs(dem.rio.crs)
    cbrfc_warped_list.append(cbrfc_warped)
    dem_cropped = dem.rio.clip(cbrfc_warped.geometry, cbrfc_warped.crs, drop=True)
    dem_cropped_list.append(dem_cropped)

# Plot them again
fig, axa = plt.subplots(1, 3, figsize=(15, 5))
for poly, ax in zip(cbrfc_warped_list, axa.flatten()):
    poly.plot(ax=ax, column='zoneNum', legend=True, cmap='viridis_r', vmin=1, vmax=3, ec='k', linewidth=0.15)

# Plot them
fig, axa = plt.subplots(1, 3, figsize=(15, 5))
for dem, ax in zip(dem_cropped_list, axa.flatten()):
    h.plot_one(dem, specify_ax=(fig, ax), title=dem.name, vmin=2000, vmax=4000, cmap='terrain')


In [ ]:
cbrfc_warped

In [ ]:
# Now clip DEM raster values by CBRFC zone
dem_zone_list = []
for dem, cbrfc in zip(dem_cropped_list, cbrfc_warped_list):
    zones = cbrfc['zone'].unique()
    dem_zone_dict = {}
    fig, axa = plt.subplots(1, len(zones), figsize=(5 * len(zones), 5))
    for zone, ax in zip(zones, axa.flatten()):
        poly = cbrfc[cbrfc['zone'] == zone]
        # Clip the raster to this zone
        dem_zone = dem.rio.clip(poly.geometry, poly.crs, drop=True)
        h.plot_one(dem_zone, specify_ax=(fig, ax), title=f'{dem.name} zone {zone}', vmin=2000, vmax=4000, cmap='terrain',
                   cbaron=False
                   )
        # Turn off axes
        h.clean_axes(ax=ax, titleoff=False, fc='lightgrey')
        # dem_zone_dict[zone] = dem_zone
        # Extract statistics
        dem_zone_dict[zone] = {
            'mean': float(dem_zone.mean().values),
            'std': float(dem_zone.std().values),
            'min': float(dem_zone.min().values),
            'max': float(dem_zone.max().values),
            'count': int(np.isfinite(dem_zone).sum().values)
        }
    # Turn dict into a dataframe
    dem_zone_df = pd.DataFrame(dem_zone_dict).T
    dem_zone_list.append(dem_zone_df)

In [ ]:
# Join the zone numbers from the CBRFC df to the DEM stats df
for dem_zone_df, cbrfc in zip(dem_zone_list, cbrfc_warped_list):
    dem_zone_df['zoneNum'] = cbrfc.set_index('zone')['zoneNum']

In [ ]:
# Filter the zones in the blue basin and group by zoneNum
blue_zones = dem_zone_list[0]
blue_zones

In [ ]:
# Group by zone and convert to feet
blue_zones.groupby('zoneNum').mean() * 100 / 2.54 / 12

In [ ]:
# Filter to just the lower zone
blue_zones[blue_zones['zoneNum'] == 1] * 100 / 2.54 / 12


In [ ]:
# Filter to just the middle zone
blue_zones[blue_zones['zoneNum'] == 2] * 100 / 2.54 / 12


In [ ]:
(blue_zones[blue_zones.index == 'DIRC2LMF'].values - blue_zones[blue_zones.index == 'DIRC2LLF'].values) * 100 / 2.54 / 12

In [ ]:
(blue_zones[blue_zones.index == 'DIRC2LUF'].values - blue_zones[blue_zones.index == 'DIRC2LMF'].values) * 100 / 2.54 / 12

In [ ]:
# Plot in 1500 ft increments
increment = 1500 * 12 * 2.54 / 100  # convert to meters
idx = 0
this_dem = dem_cropped_list[idx]

fig, axa = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
ax = axa[0]
h.plot_one(this_dem, specify_ax=(fig, ax), title=this_dem.name, vmin=np.nanmin(this_dem.values), vmax=np.nanmax(this_dem.values), cmap='terrain')

# Plot contours by increments on top
this_dem.plot.contour(ax=ax, levels=np.arange(6500 * 12 * 2.54 / 100, 11000 * 12 * 2.54 / 100, increment), colors='k', linewidths=0.75)
ax.set_title(f'{this_dem.name}')

# Fill the contours with different colors and remove colorbar
ax = axa[1]
this_dem.plot.contourf(ax=ax, levels=np.arange(6500 * 12 * 2.54 / 100, 11000 * 12 * 2.54 / 100, increment), cmap='viridis_r', alpha=1)
# Plot outlines of CBRFC zones
cbrfc_warped_list[idx].plot(ax=ax, column='zoneNum', legend=False, cmap='viridis_r', facecolor='None', vmin=1, vmax=3, ec='k', linewidth=0.25)
h.clean_axes(ax=ax, fc='white', ticksoff=False, gridon=False)
ax.set_title(f'{this_dem.name} contours every {int(increment*100/2.54/12)} ft')

# Plot the CBRFC zones alongside
ax = axa[2]
zone_ax = cbrfc_warped_list[idx].plot(ax=ax, column='zoneNum', legend=False, cmap='viridis_r', vmin=1, vmax=3, ec='k', linewidth=0.15)
zones = zone_ax.collections[0]
ax.set_title('CBRFC zones')
# Change legend label, convert to discrete integer values, shrink the colorbar
plt.colorbar(zones, ax=ax, label='CBRFC zone number', ticks=[1, 2, 3], aspect=25, shrink=0.75, pad=0.02)

In [ ]:
# Plot in 1500 ft increments
increment = 1500 * 12 * 2.54 / 100  # convert to meters
idx = 1
this_dem = dem_cropped_list[idx]

fig, axa = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
ax = axa[0]
h.plot_one(this_dem, specify_ax=(fig, ax), title=this_dem.name, vmin=np.nanmin(this_dem.values), vmax=np.nanmax(this_dem.values), cmap='terrain')

# Plot contours by increments on top
this_dem.plot.contour(ax=ax, levels=np.arange(6500 * 12 * 2.54 / 100, 11000 * 12 * 2.54 / 100, increment), colors='k', linewidths=0.75)
ax.set_title(f'{this_dem.name}')

# Fill the contours with different colors and remove colorbar
ax = axa[1]
this_dem.plot.contourf(ax=ax, levels=np.arange(6500 * 12 * 2.54 / 100, 11000 * 12 * 2.54 / 100, increment), cmap='viridis_r', alpha=1)
# Plot outlines of CBRFC zones
cbrfc_warped_list[idx].plot(ax=ax, column='zoneNum', legend=False, cmap='viridis_r', facecolor='None', vmin=1, vmax=3, ec='k', linewidth=0.25)
h.clean_axes(ax=ax, fc='white', ticksoff=False, gridon=False)
ax.set_title(f'{this_dem.name} contours every {int(increment*100/2.54/12)} ft')

# Plot the CBRFC zones alongside
ax = axa[2]
zone_ax = cbrfc_warped_list[idx].plot(ax=ax, column='zoneNum', legend=False, cmap='viridis_r', vmin=1, vmax=3, ec='k', linewidth=0.15)
zones = zone_ax.collections[0]
ax.set_title('CBRFC zones')
# Change legend label, convert to discrete integer values, shrink the colorbar
plt.colorbar(zones, ax=ax, label='CBRFC zone number', ticks=[1, 2, 3], aspect=25, shrink=0.75, pad=0.02)

In [ ]:
# Plot in 1500 ft increments
idx = 2
this_dem = dem_cropped_list[idx]

fig, axa = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
ax = axa[0]
h.plot_one(this_dem, specify_ax=(fig, ax), title=this_dem.name, vmin=np.nanmin(this_dem.values), vmax=np.nanmax(this_dem.values), cmap='terrain')

# Plot contours by increments on top
this_dem.plot.contour(ax=ax, levels=np.arange(6500 * 12 * 2.54 / 100, 11000 * 12 * 2.54 / 100, increment), colors='k', linewidths=0.75)
ax.set_title(f'{this_dem.name}')

# Fill the contours with different colors and remove colorbar
ax = axa[1]
this_dem.plot.contourf(ax=ax, levels=np.arange(6500 * 12 * 2.54 / 100, 11000 * 12 * 2.54 / 100, increment), cmap='viridis_r', alpha=1)
# Plot outlines of CBRFC zones
cbrfc_warped_list[idx].plot(ax=ax, column='zoneNum', legend=False, cmap='viridis_r', facecolor='None', vmin=1, vmax=3, ec='k', linewidth=0.25)
h.clean_axes(ax=ax, fc='white', ticksoff=False, gridon=False)
ax.set_title(f'{this_dem.name} contours every {int(increment*100/2.54/12)} ft')

# Plot the CBRFC zones alongside
ax = axa[2]
zone_ax = cbrfc_warped_list[idx].plot(ax=ax, column='zoneNum', legend=False, cmap='viridis_r', vmin=1, vmax=3, ec='k', linewidth=0.15)
zones = zone_ax.collections[0]
ax.set_title('CBRFC zones')
# Change legend label, convert to discrete integer values, shrink the colorbar
plt.colorbar(zones, ax=ax, label='CBRFC zone number', ticks=[1, 2, 3], aspect=25, shrink=0.75, pad=0.02)

In [ ]:
# Group elevation by zone numbers and plot the distribution as boxplots
for v in ['min', 'mean', 'max']:
    fig, axa = plt.subplots(1, len(basins), figsize=(4*len(basins), 3), sharey=True)
    for dem_zone_df, basin, ax in zip(dem_zone_list, basins, axa.flatten()):
        # Remove entries with zonenum == 0
        dem_zone_df = dem_zone_df[dem_zone_df['zoneNum'] != 0]
        sns.boxplot(dem_zone_df, x='zoneNum', y=v, ax=ax, palette='icefire', width=0.25)
        ax.set_title(f'{basin.capitalize()} {v} elevation by CBRFC zone')
        ax.grid('lightgrey', linestyle='--')
    plt.suptitle('')

In [ ]:
# Now plot all the basin boxplots on a single plot for comparison
# Ordering for zonenum 1: Blue, Animas, Yampa
vars = ['min', 'mean', 'max']
adj = 200
fig, axa = plt.subplots(1, len(vars), figsize=(6*len(vars), 4), sharey=True)
for jdx, (v, ax) in enumerate(zip(vars, axa.flatten())):
    # fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    for dem_zone_df, basin in zip(dem_zone_list, basins):
        # Remove entries with zonenum == 0
        dem_zone_df = dem_zone_df[dem_zone_df['zoneNum'] != 0]
        sns.boxplot(dem_zone_df, x='zoneNum', y=v, ax=ax, width=0.15,
                    palette=sns.set_palette('icefire'),
                    label=basin.capitalize(), positions=np.arange(3) + (basins.index(basin)-1)*0.25)
        # Add the number of samples (rows in the df for this zoneNum above the boxplot
        for zoneNum in dem_zone_df['zoneNum'].unique():
            pix_count = dem_zone_df[dem_zone_df['zoneNum'] == zoneNum]['count'].values
            unit_count = len(pix_count)
            q3 = dem_zone_df[v].quantile(0.75)
            top = 4250
            # ax.text(zoneNum - 1 + (basins.index(basin)-1)*0.25, dem_zone_df[v].max() + 50, str(unit_count),
            ax.text(zoneNum - 1 + (basins.index(basin)-1)*0.25, top, str(unit_count),
                    horizontalalignment='center', size='small', color='k', weight='semibold',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1))
            # Add the pixel count beneath this annotation
            total_pix = dem_zone_df[dem_zone_df['zoneNum'] == zoneNum]['count'].sum()
            ax.text(zoneNum - 1 + (basins.index(basin)-1)*0.25, top - adj*(jdx+1), f'({total_pix:.0f} pix)',
            # ax.text(zoneNum - 1 + (basins.index(basin)-1)*0.25, dem_zone_df[v].max() - 75, f'({total_pix:.0f} pix)',
                    verticalalignment='top', horizontalalignment='center', size=8, color='k', rotation=90,
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1)
                    )
            # Also add the range of elevations in this zone
            elev_min = dem_zone_df[dem_zone_df['zoneNum'] == zoneNum][v].min()
            elev_max = dem_zone_df[dem_zone_df['zoneNum'] == zoneNum][v].max()
            ax.text(zoneNum - 1 + (basins.index(basin)-1)*0.25, top + 400, f'[{elev_min:.0f}, {elev_max:.0f}]',
                    verticalalignment='bottom', horizontalalignment='center', size=9, color='k', rotation=90,
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1)
                    )
        ax.grid('lightgrey', linestyle='--')
    # Add lines at 2800 and 3600 m
    ax.axhline(2800, color='k', linestyle=':', label='2800 m')
    ax.axhline(3600, color='k', linestyle=':', label='3600 m')
    # Rename xticks  low, middle and upper
    ax.set_xticks(np.arange(3), labels=['Lower', 'Middle', 'Upper'])
    ax.set_xlabel('Modeling Unit Elevation Zone')
    ax.set_ylabel(f'{v.capitalize()} Elevation (m)')
    # plt.suptitle(f'CBRFC zone {v.capitalize()} elevation comparison')
# plt.suptitle(f'CBRFC zone elevation comparison')

In [ ]:
# Calculate stats for each basin and zone
cbrfc_zone_group_list = []
for dem_zone_df, basin in zip(dem_zone_list, basins):
    # Remove entries with zonenum == 0
    dem_zone_df = dem_zone_df[dem_zone_df['zoneNum'] != 0]
    # Calculate summed pixel counts and percentages of all pixels in the dataframe (basin)
    dem_zone_df['pix_pct'] = dem_zone_df['count'] / dem_zone_df['count'].sum() * 100
    # Group by zoneNum
    cbrfc_zone_grouped = dem_zone_df.groupby('zoneNum').sum()[['count', 'pix_pct']]
    # Add basin name as df name
    cbrfc_zone_grouped.name = basin.capitalize()
    cbrfc_zone_group_list.append(cbrfc_zone_grouped)

In [ ]:
for df in cbrfc_zone_group_list:
    print(df.name)
    print(df)
    print('\n')

In [ ]:
# For each dem_cropped basin, determine the percentage of pixels in the following zones
zone_bounds = {
    'low': (0, 2800),
    'mid': (2800, 3600),
    'high': (3600, 10000)
}
zone_perc_list = []
for dem, basin in zip(dem_cropped_list, basins):
    total_pix = np.isfinite(dem).sum().values
    zone_perc_dict = {}
    for zone, (zmin, zmax) in zone_bounds.items():
        zone_pix = ((dem >= zmin) & (dem < zmax)).sum().values
        zone_perc = zone_pix / total_pix * 100
        zone_perc_dict[zone] = {
            'pix_count': int(zone_pix),
            'percentage': float(zone_perc)
        }
    zone_perc_df = pd.DataFrame(zone_perc_dict).T
    zone_perc_list.append(zone_perc_df)
# Combine into a single dataframe
zone_perc_all = pd.concat(zone_perc_list, keys=basins, names=['basin', 'zone'])
zone_perc_all = zone_perc_all.reset_index()

# Plot it by basin and zone
zone_perc_all['zone'] = pd.Categorical(zone_perc_all['zone'], categories=['low', 'mid', 'high'], ordered=True)
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
sns.barplot(zone_perc_all, x='basin', y='percentage', hue='zone', ax=ax, palette='rocket')
# Annotate percentages above the bars
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(p.get_x() + p.get_width() / 2., height + 1,
                f'{height:.0f}%', ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, 100)
ax.set_ylabel('Percentage of Basin Area (%)')
# Add the zone bounds as the title
plt.suptitle(f'Basin Area Percentage by Elevation Zone\n(Low: <{zone_bounds['low'][1]} m, Mid: {zone_bounds['mid'][0]}-{zone_bounds['mid'][1]} m, High: >{zone_bounds['high'][0]} m)')


In [ ]:
# dem_cropped_list[0].plot.imshow(vmin=2500, vmax=4300)
# Plot only values less than 2700 m
dem_cropped_list[0].plot.imshow(vmin=2000, vmax=4300, cmap='terrain', cbar_kwargs={'label': 'Elevation (m)'})
# dem_cropped_list[0].where(dem_cropped_list[0] < 3000).plot.imshow(vmin=2000, vmax=4300, cmap='terrain', cbar_kwargs={'label': 'Elevation (m)'})

In [ ]:
# Calculate the entire area of the yampa basin
yampa_gdf = cbrfc_warped_list[-1]
yampa_gdf


In [ ]:
# Dissolve polygons into single geom
dissolved_geometry = yampa_gdf.dissolve(by=None).geometry.squeeze()
dissolved_geometry

In [ ]:
from shapely.geometry import Polygon, MultiPolygon
from shapely import unary_union

In [ ]:
# Fill holes
if isinstance(dissolved_geometry, (Polygon, MultiPolygon)):
    # If the result is a MultiPolygon, process each part
    if isinstance(dissolved_geometry, MultiPolygon):
        filled_polygons = [Polygon(p.exterior.coords) for p in dissolved_geometry.geoms]
        final_geometry = unary_union(filled_polygons)
    # If the result is a single Polygon, extract its exterior
    else:
        final_geometry = Polygon(dissolved_geometry.exterior.coords)

final_geometry

In [ ]:
# in square meters, convert to square kilometers
final_geometry.area / 1e6